# Wildlife Camera EDA

***First-pass exploratory analysis for the SDRPF wildlife camera dataset. This notebook looks at data quality, species composition, camera/preserve coverage, seasonality, daily activity patterns, and 30-minute independent events.***

## Setup

If imports fail, install the common EDA stack in the notebook kernel first:

```python
%pip install pandas numpy matplotlib seaborn
```

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)

DATA_PATH = Path("data/Derek Master Sheet 2012-2018 - All Data.csv")

: 

## Load And Clean

In [ ]:
raw = pd.read_csv(DATA_PATH)

df = raw.copy()
df.columns = df.columns.str.strip()

text_cols = df.select_dtypes(include="object").columns
df[text_cols] = df[text_cols].apply(lambda s: s.str.strip())

df["timestamp_raw"] = df["timestamp"]
try:
    df["timestamp"] = pd.to_datetime(df["timestamp_raw"], errors="coerce", format="mixed")
except TypeError:
    df["timestamp"] = pd.to_datetime(df["timestamp_raw"], errors="coerce")

for col in ["year", "month", "day"]:
    df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

df["date"] = df["timestamp"].dt.date
df["hour"] = df["timestamp"].dt.hour
df["weekday"] = df["timestamp"].dt.day_name()
df["week"] = df["timestamp"].dt.to_period("W").dt.start_time
df["year_month"] = df["timestamp"].dt.to_period("M").dt.to_timestamp()
df["species_label"] = df["common_name"].fillna("Unknown").replace("", "Unknown")

season_map = {
    12: "Winter", 1: "Winter", 2: "Winter",
    3: "Spring", 4: "Spring", 5: "Spring",
    6: "Summer", 7: "Summer", 8: "Summer",
    9: "Fall", 10: "Fall", 11: "Fall",
}
df["season"] = df["month"].map(season_map)

df["day_part"] = pd.Series(pd.NA, index=df.index, dtype="object")
df.loc[df["hour"].between(5, 8), "day_part"] = "Dawn"
df.loc[df["hour"].between(9, 16), "day_part"] = "Day"
df.loc[df["hour"].between(17, 20), "day_part"] = "Dusk"
df.loc[df["hour"].notna() & df["day_part"].isna(), "day_part"] = "Night"

df.head()

In [ ]:
overview = pd.Series({
    "rows": len(df),
    "columns": df.shape[1],
    "date_min": df["timestamp"].min(),
    "date_max": df["timestamp"].max(),
    "preserves": df["preserve"].nunique(dropna=True),
    "cameras": df["camera"].nunique(dropna=True),
    "deployments": df["deployment_id"].nunique(dropna=True),
    "common_names": df["species_label"].nunique(dropna=True),
    "exact_duplicate_rows": df.duplicated().sum(),
    "unparsed_timestamps": df["timestamp"].isna().sum(),
})
overview.to_frame("value")

## Data Quality Checks

In [ ]:
missing = (
    df.isna().sum()
    .to_frame("missing")
    .assign(pct=lambda x: x["missing"] / len(df))
    .sort_values("missing", ascending=False)
)
missing.style.format({"pct": "{:.1%}"})

In [ ]:
duplicate_examples = df[df.duplicated(keep=False)].sort_values(
    ["timestamp", "camera", "species_label"]
)

print(f"Exact duplicate rows: {df.duplicated().sum():,}")
duplicate_examples.head(20)

In [ ]:
records_by_year = df["year"].value_counts().sort_index()
records_by_year.to_frame("records")

## Independent Events

Camera trap datasets often contain bursts of photos from the same animal visit. The helper below creates 30-minute independent events by `camera` and `species_label`. You can change `EVENT_GAP_MINUTES` if your project uses a different standard.

In [ ]:
EVENT_GAP_MINUTES = 30

events = df.dropna(subset=["timestamp"]).copy()
events = events.sort_values(["camera", "species_label", "timestamp"])

gap = events.groupby(["camera", "species_label"])["timestamp"].diff()
events["new_event"] = gap.isna() | (gap > pd.Timedelta(minutes=EVENT_GAP_MINUTES))
events["event_id"] = events.groupby(["camera", "species_label"])["new_event"].cumsum()

event_summary = (
    events.groupby(["camera", "species_label", "event_id"], as_index=False)
    .agg(
        preserve=("preserve", "first"),
        deployment_id=("deployment_id", "first"),
        latitude=("latitude", "first"),
        longitude=("longitude", "first"),
        event_start=("timestamp", "min"),
        event_end=("timestamp", "max"),
        photos=("timestamp", "size"),
    )
)
event_summary["duration_minutes"] = (
    event_summary["event_end"] - event_summary["event_start"]
).dt.total_seconds() / 60
event_summary["year"] = event_summary["event_start"].dt.year
event_summary["month"] = event_summary["event_start"].dt.month
event_summary["hour"] = event_summary["event_start"].dt.hour
event_summary["season"] = event_summary["month"].map(season_map)

print(f"Raw detections with timestamps: {len(events):,}")
print(f"Independent {EVENT_GAP_MINUTES}-minute events: {len(event_summary):,}")
event_summary.head()

## Species Composition

In [ ]:
top_species = df["species_label"].value_counts().head(20)
top_event_species = event_summary["species_label"].value_counts().head(20)

fig, axes = plt.subplots(1, 2, figsize=(15, 7), sharey=False)
sns.barplot(x=top_species.values, y=top_species.index, ax=axes[0], color="#4477AA")
axes[0].set_title("Top Common Names: Raw Detections")
axes[0].set_xlabel("Detections")
axes[0].set_ylabel("")

sns.barplot(x=top_event_species.values, y=top_event_species.index, ax=axes[1], color="#228833")
axes[1].set_title(f"Top Common Names: {EVENT_GAP_MINUTES}-Minute Events")
axes[1].set_xlabel("Events")
axes[1].set_ylabel("")

plt.tight_layout()

In [ ]:
species_table = (
    df.groupby("species_label")
    .agg(
        detections=("species_label", "size"),
        preserves=("preserve", "nunique"),
        cameras=("camera", "nunique"),
        first_seen=("timestamp", "min"),
        last_seen=("timestamp", "max"),
    )
    .join(event_summary["species_label"].value_counts().rename("events_30m"))
    .fillna({"events_30m": 0})
    .sort_values("detections", ascending=False)
)
species_table.head(25)

## Camera And Preserve Coverage

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

preserve_counts = df["preserve"].value_counts()
sns.barplot(x=preserve_counts.index, y=preserve_counts.values, ax=axes[0], color="#66CCEE")
axes[0].set_title("Raw Detections By Preserve")
axes[0].set_xlabel("Preserve")
axes[0].set_ylabel("Detections")

camera_counts = df["camera"].value_counts().head(20)
sns.barplot(x=camera_counts.values, y=camera_counts.index, ax=axes[1], color="#CCBB44")
axes[1].set_title("Top Cameras By Raw Detections")
axes[1].set_xlabel("Detections")
axes[1].set_ylabel("Camera")

plt.tight_layout()

In [ ]:
camera_locations = (
    df.groupby(["preserve", "camera", "latitude", "longitude"], as_index=False)
    .agg(detections=("species_label", "size"), species=("species_label", "nunique"))
)

plt.figure(figsize=(8, 7))
sns.scatterplot(
    data=camera_locations,
    x="longitude",
    y="latitude",
    hue="preserve",
    size="detections",
    sizes=(50, 500),
    alpha=0.8,
)
plt.title("Camera Locations Sized By Detections")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()

## Trends Over Time

In [ ]:
monthly = df.dropna(subset=["year_month"]).groupby("year_month").size().rename("detections")

plt.figure(figsize=(14, 4))
monthly.plot(color="#4477AA")
plt.title("Monthly Raw Detections")
plt.xlabel("")
plt.ylabel("Detections")
plt.tight_layout()

In [ ]:
top10 = df["species_label"].value_counts().head(10).index
species_year = (
    df[df["species_label"].isin(top10)]
    .pivot_table(index="year", columns="species_label", values="timestamp_raw", aggfunc="count", fill_value=0)
    .sort_index()
)

plt.figure(figsize=(14, 6))
sns.lineplot(data=species_year, markers=True, dashes=False)
plt.title("Top Species Detections By Year")
plt.xlabel("Year")
plt.ylabel("Detections")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()

In [ ]:
month_order = list(range(1, 13))
monthly_species = (
    df[df["species_label"].isin(top10)]
    .pivot_table(index="species_label", columns="month", values="timestamp_raw", aggfunc="count", fill_value=0)
    .reindex(columns=month_order)
)

plt.figure(figsize=(12, 6))
sns.heatmap(monthly_species, cmap="YlGnBu", linewidths=0.5)
plt.title("Seasonality Heatmap For Top Species")
plt.xlabel("Month")
plt.ylabel("")
plt.tight_layout()

## Daily Activity Patterns

In [ ]:
hourly_events = (
    event_summary[event_summary["species_label"].isin(top10)]
    .pivot_table(index="species_label", columns="hour", values="event_id", aggfunc="count", fill_value=0)
    .reindex(columns=range(24), fill_value=0)
)

plt.figure(figsize=(14, 6))
sns.heatmap(hourly_events, cmap="mako", linewidths=0.5)
plt.title(f"Hourly Activity For Top Species ({EVENT_GAP_MINUTES}-Minute Events)")
plt.xlabel("Hour Of Day")
plt.ylabel("")
plt.tight_layout()

In [ ]:
day_part_order = ["Dawn", "Day", "Dusk", "Night"]
day_part = (
    df[df["species_label"].isin(top10)]
    .pivot_table(index="species_label", columns="day_part", values="timestamp_raw", aggfunc="count", fill_value=0)
    .reindex(columns=day_part_order)
)

day_part_pct = day_part.div(day_part.sum(axis=1), axis=0)
day_part_pct.plot(kind="barh", stacked=True, figsize=(12, 6), colormap="viridis")
plt.title("Share Of Detections By Day Part")
plt.xlabel("Share")
plt.ylabel("")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()

## Starter Questions To Pursue

- Which species are widespread across preserves versus concentrated at a few cameras?
- Do raw detection counts tell a different story than 30-minute independent events?
- Are there years or months with sharp changes that reflect camera effort rather than wildlife activity?
- Which species are mainly nocturnal, diurnal, or crepuscular?
- Which taxonomy gaps should be reviewed manually, especially common names such as family-level or generic labels?

In [ ]:
# Optional exports after reviewing the cleaning choices above.
# Path("outputs").mkdir(exist_ok=True)
# df.to_csv("outputs/wildlife_camera_cleaned.csv", index=False)
# event_summary.to_csv("outputs/wildlife_camera_events_30m.csv", index=False)